# Incomplete-Info-Problem — quick experiments notebook

This notebook is a thin wrapper over the new `iip` API. For a full overview see the README.

**Workflow:**
1. Train a small Deep CFR agent on HULHE.
2. Evaluate vs baselines (mbb/h) + LBR exploitability.
3. (Optional) Fine-tune with PPO self-play.

In [ ]:
from iip.engine.game import HULHE
from iip.train.deepcfr_trainer import DeepCFRTrainer, DeepCFRTrainerConfig
from iip.train.game_adapter import HULHEAdapter

adapter = HULHEAdapter(game=HULHE())
cfg = DeepCFRTrainerConfig(traversals_per_iteration=500, advantage_train_steps=300, strategy_train_steps=600, batch_size=128)
trainer = DeepCFRTrainer(adapter=adapter, config=cfg, seed=0)
trainer.iterate(n_iters=3)
agent = trainer.finalize_strategy()
agent.save('checkpoints/local/deepcfr_hulhe.pt')

In [ ]:
import random
from iip.agents.fixed_policy import FishAgent, StrengthHandAgent
from iip.agents.random_agent import RandomAgent
from iip.metrics.mbb import head_to_head_mbb
from iip.metrics.exploitability import local_best_response

for opp in [FishAgent(), StrengthHandAgent(), RandomAgent(rng=random.Random(0))]:
    print(head_to_head_mbb(adapter.game, agent, opp, n_hands=500, seed=0))

print('LBR mbb/h:', local_best_response(adapter.game, agent, n_hands=300, seed=0))

In [ ]:
from iip.agents.ppo import PPOAgent, PPOConfig
from iip.train.ppo_trainer import PPOTrainer

ppo = PPOAgent(feature_dim=adapter.feature_dim, num_actions=adapter.num_actions)
ppo.warm_start_from_deepcfr(agent.strategy_net.state_dict())
trainer_ppo = PPOTrainer(
    adapter=adapter,
    agent=ppo,
    config=PPOConfig(actor_hidden=[256, 256], critic_hidden=[256, 256]),
    opponents=[FishAgent(), StrengthHandAgent(), RandomAgent(rng=random.Random(0))],
)
for i in range(3):
    stats = trainer_ppo.train_round(n_hands=500)
    print(i, stats)
ppo.save('checkpoints/local/ppo_hulhe.pt')